# Iceberg layer review -- silver and gold

Read-only look at the **Iceberg** layers, through **Spark**.

Why not pandas, the way `review_layers.ipynb` reads raw and quarantine? An
Iceberg table is a set of parquet files *plus metadata deciding which of them
are live*. Both `merge` and `delete_insert` rewrite files, so reading the
parquet directly would show rows a later snapshot deleted -- silently, with no
hint that the numbers are wrong. Spark is the authoritative reader.

The helpers live in `review_gold.py` beside this notebook and build every
identifier from `config/local.yaml`, so this cannot drift from where the jobs
actually write.

**The session costs ~30s to start** (JVM, then Ivy resolving the Iceberg
jars). The kernel holds it across every cell, so you pay that once -- which is
the whole reason this is a notebook and not a script.

In [ ]:
# Pick up edits to review_gold.py without restarting the kernel. A restart
# would also drop the Spark session, and rebuilding it means another cold
# start. Without this, adding a helper to the .py leaves the kernel holding
# the version it imported first -- an AttributeError that looks like the
# function was never written.
%load_ext autoreload
%autoreload 2

import sys

import pandas as pd

sys.path.insert(0, "..")   # the python_scripts package, from the repo root
sys.path.insert(0, ".")    # review_gold.py, beside this notebook

import review_gold as rg

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

spark, cfg = rg.connect("prod")      # ~30s on a cold kernel
print(spark.conf.get("spark.sql.catalog.iceberg.warehouse"))
print(f"gold tables in config: {sorted(cfg.gold)}")

## 1. What exists

Start here after any run. Row counts come from the Iceberg **manifests**, not
from scanning data, so this stays cheap however large the tables get.

A table listed with `rows = 0` means a job created it and wrote an empty
frame -- a different failure from the job never running, and worth telling
apart.

If `gold` comes back empty, no gold job has run against *this* warehouse yet.

In [ ]:
rg.namespaces(spark)

In [ ]:
rg.inventory(spark, cfg, "gold")

In [ ]:
rg.inventory(spark, cfg, "silver")

## 2. The silver tables

`silver_overview` is the one call to run after a load. It is driven by the
`tables:` contracts in config, so adding a dataset picks it up with no change
here.

Read `rows` against `distinct_keys` — and what a gap means depends on the
write strategy:

| strategy | expected | a gap means |
|---|---|---|
| `merge` (order_items) | equal | rows were quarantined, not merged — Iceberg refuses a MERGE where two source rows hit one target row |
| `delete_insert` (order_item_options) | rows **may** exceed | nothing: that table has no unique key of its own, which is why it is scoped by `ORDER_ID` |
| `replace` (date_dim) | equal | the rewrite dropped rows |

In [ ]:
rg.silver_overview(spark, cfg)

### Did the configured casts actually land?

`required_columns` describes the *source* types; `silver_transforms` overrides
them on the way in. This is where you confirm that happened — a string
`creation_time_utc` here would mean the cast silently didn't apply, and every
downstream `to_date()` would be operating on text.

Expect `timestamp` for `creation_time_utc` and `decimal(10,2)` for the two
price columns.

In [ ]:
import pandas as pd

pd.DataFrame([
    {"table": t, "column": c, "type": d}
    for t in ("date_dim", "order_items", "order_item_options")
    for c, d in spark.table(rg.fqn(cfg, "silver", t)).dtypes
]).pivot_table(index="column", columns="table", values="type", aggfunc="first")

### Partitioning

Only checkable here, because Iceberg applies the partition spec at table
*creation*. `order_items` should show one partition per month of event time —
not 1336 daily ones, which is what `days()` produced and the Readme's
small-file note explains.

`date_dim` and `order_item_options` are unpartitioned, so they return a single
row covering the whole table.

In [ ]:
rg.partitions(spark, cfg, "silver", "order_items")

### Referential integrity across the two order tables

Option rows whose parent line item is missing from `order_items`. The extract
widens to whole parents for `delete_insert` tables, so an option can only
arrive alongside its siblings -- which means a *load* failure would show up
here in bulk.

It is not empty, and that is a known source condition rather than a pipeline
fault: **28 option rows across 14 orders**, whose `order_id` is absent from
`order_items` entirely -- not a line item missing from a present order. So the
options reference orders that were never in the source extract. It is also not
the one quarantined row: no option row has a null `lineitem_id`.

What to watch for is the number moving. 28 is the baseline; a jump means the
parent-widening broke or the two tables were loaded for different periods.

In [ ]:
rg.orphan_options(spark, cfg)

### A few rows of each

In [ ]:
for t in ("date_dim", "order_items", "order_item_options"):
    print(f"--- {t} ---")
    display(rg.peek(spark, cfg, "silver", t, limit=5))

## 3. The gold tables

Seven of them, in three kinds:

| table | grain | rebuilt | what it answers |
|---|---|---|---|
| `daily_restaurant_sales` | date x restaurant | incrementally | how each site trades day to day |
| `restaurant_revenue_rank` | restaurant | in full | which sites lead over all history |
| `customer_lifetime_value` | customer | in full | who the valuable customers are |
| `customer_rfm` | customer | in full | who is active, lapsing or gone |
| `category_period_sales` | period_type x period x category | in full | how categories trade by week, month, season |
| `category_holiday_sales` | holiday x category | in full | how holidays trade against an ordinary day |
| `loyalty_order_profile` | population x membership_state | in full | whether the loyalty program changes behaviour |

Only `daily_restaurant_sales` is incremental. The other six are `replace`, for
the same reason each time: a rank, a percentile, a lifetime total and a
before/after split cannot be computed from one batch's slice of rows.

`gold_overview` is the call to run after a silver_to_gold run. Like
`silver_overview` it is driven by config -- `gold:` -- so registering the next
transform adds it here with no edit to the notebook or the helper module.

In [ ]:
rg.schema(spark, cfg, "gold", "daily_restaurant_sales")

In [ ]:
rg.peek(spark, cfg, "gold", "restaurant_revenue_rank",
        limit=15, order_by="revenue_rank")

In [ ]:
rg.leaderboard(spark, cfg)

### Is every grain still unique?

`rows` must equal `distinct_grain` for every gold table. Silver has write
strategies where a gap is legitimate; gold has none. `grain_keys` is what makes
a gold row unique, so a gap means the transform emits duplicates and every
join against the table will fan out.

In [ ]:
rg.gold_overview(spark, cfg)

### Does each table still publish what it promises?

`output_columns` is an *output* contract -- the opposite direction to
`required_columns`, which guards the input from raw. It is deliberately a
subset: a transform may emit more than it declares, so an extra column is not
a finding and an additive change stays a one-place edit.

The two things that *are* findings: a declared column that stopped being
emitted (`MISSING`), and one whose type drifted (`WRONG TYPE`). Either breaks
a consumer silently, which is the whole argument for checking it here rather
than noticing in a dashboard.

In [ ]:
contract = rg.output_contract(spark, cfg)
print(contract.status.value_counts().to_string())

contract[contract.status != "ok"]      # empty frame is the passing result

### `loyalty_order_profile` -- five rows that must not be summed

The five rows **deliberately overlap**. A switcher's flagged order is counted
in three of them, so `sum(total_revenue)` over this table reports roughly 2.4x
the business. They are five separate answers, not a partition.

Two different comparisons live here, and telling them apart is the point:

* **Never members vs All members** is a LEVEL difference, and is confounded by
  who chooses to enrol.
* **Observed in both states**, Pre-member vs Member, is a CHANGE within the
  same 2,783 customers, and is not.

`orders_per_customer` is derived in the helper rather than stored. Read it
*across* populations. Do not read it between Pre-member and Member -- two cells
down explains why not.

In [ ]:
rg.loyalty_profile(spark, cfg)

#### "All members" averages two populations that behave nothing alike

Which is why it is the weakest row in the table to quote. Split the identified
customers three ways and the member side comes apart: customers flagged on
*every* order order about as often as people who never joined, while switchers
order roughly six times as much. Pooling them yields a number that describes
neither group.

In [ ]:
rg.loyalty_cohorts(spark, cfg)

#### Orders per customer is a count, so it needs a time denominator

Pre-member shows about 13.2 orders per customer against Member's 7.4, which
looks like joining halved order frequency. It did not. Those are the same 2,783
customers, observed un-enrolled for ~206 days on average and enrolled for
~117 -- the count falls because the second window is shorter, not because
behaviour changed.

Dividing by observed days settles it, and the two rates come out flat. So the
honest answer to "does the program raise order frequency" is **no measurable
change**, and the apparent change is an exposure artifact. That is also why the
gold table stores per-order ratios and no counts-per-customer: every stored
measure is already normalised, and this one cannot be.

One caveat bounds where the rate is readable. A customer with a single order in
a state spans one day, so they contribute one order per day and drag the pooled
rate up. Switchers average 7 and 13 orders per state so it barely matters here;
for never members, who average 3.5, it would.

In [ ]:
rg.loyalty_order_rate(spark, cfg)

#### Does "Pre-member" actually mean "before"?

Worth being explicit that the whole before/after reading rests on an
assumption. `is_loyalty` is a line-item flag and the source carries no
enrolment date, so "Pre-member" strictly means "unflagged" -- and the reading
only holds if a switcher's unflagged orders all precede their flagged ones.

If the flag instead records whether loyalty was *applied* to an order, a member
who forgets their card lands in Pre-member and contaminates the before group.
This cell is what tells those two worlds apart, and the answer belongs in the
Readme rather than being left implicit.

In [ ]:
rg.loyalty_flag_ordering(spark, cfg)

#### Frequency bias -- why the cohort gap is not a finding

`loyalty_cohorts()` above puts switchers at ~20.6 orders per customer against
~3.5 for both other cohorts. A six-fold gap is the kind of number that ends up
on a slide, so it is worth being exact about what it can support.

The problem is the definition, not the data. A switcher is a customer with at
least one flagged **and** one unflagged order. That label cannot be earned
without ordering at least twice -- and the more someone orders, the more
chances they get to display both states. So **order count partly causes the
label**, which makes "switchers order more" close to circular: it approaches
saying that customers with many orders have many orders.

This is selection on the outcome, and it is worth recognising by name because
it recurs whenever a group is defined by something that accumulates with
activity -- "engaged users", "multi-category shoppers", "customers who tried
two features". Every one of those needs more activity to qualify, so every one
will look more active than the group that failed to qualify, no matter what the
program did.

Three cells to separate the artifact from the effect: measure the bias, then
hold order count fixed, then find a split that does not depend on order count
at all.

In [ ]:
rg.loyalty_cohort_spread(spark, cfg)

`one_order_customers` is the tell. It is **0** for switchers -- impossible by
construction -- while over half of each other cohort sits at exactly one
order. So the 3.5 figures are largely one-order customers and the 20.6
structurally excludes them. Those are not comparable populations, and the
medians (9 against 1 and 1) say the same thing.

Now the bias measured rather than argued: cohort share within each
order-count band.

In [ ]:
rg.loyalty_frequency_bias(spark, cfg)

`pct_switcher` climbs 0% -> 9% -> 16.5% -> 33% -> 50% -> 66% -> 70%,
monotonically, with nothing changing but order count. Among one-order
customers the label is impossible; among 50+ order customers a varying flag is
almost unavoidable. Treat that column as a warning label on the cohort gap.

So hold order count fixed. Inside a single band every cohort has roughly the
same `mean_orders`, the mechanical advantage is gone, and frequency cannot
differ -- which is the point. What is left is spend, and that comparison is
honest.

In [ ]:
rg.loyalty_matched_bands(spark, cfg)

This is where a real effect shows up. **"Flagged on every order" has the lowest
`avg_order_value` in every band**, and "never flagged" the highest in four of
five. Holding at every frequency level is what makes it an effect rather than a
composition artifact -- and it is the defensible version of the
price-sensitivity reading.

Ignore the `20+` row for matching purposes: it spans 20 to 2,124 orders, so
`mean_orders` genuinely differs inside it and it is not a real match.

One split remains that has no order-count dependence whatsoever -- classify
each customer by their **first order alone**. A single order decides the label,
so nobody can earn it by ordering more. It is the closest thing here to a fair
test of "does starting on loyalty lead to ordering more?"

In [ ]:
rg.loyalty_first_order(spark, cfg)

#### The decisive test: what would a meaningless flag produce?

The argument so far is that order count causes the switcher label. That is more
convincing measured than asserted, and it can be measured exactly.

Keep the real distribution of orders per customer untouched, and replace the
loyalty flag with a biased coin whose probability is the observed flagged share
(p = 0.264). The flag now has **zero** relationship to behaviour by
construction. For a customer with n orders:

$$P(\text{switcher} \mid n) = 1 - p^n - (1-p)^n$$

because a non-switcher is either all-flagged or none-flagged. Exact, no
simulation, and it needs only the real histogram of n.

If a meaningless flag still produces a large frequency gap, the observed gap
tells us nothing about the program.

In [ ]:
rg.loyalty_null_model(spark, cfg)

**The null produces a bigger gap than reality does** -- 9.85x against the 5.86x
actually observed. A flag with no behavioural meaning whatsoever fully accounts
for the frequency difference, and then some.

So this is not "mostly an artifact" but entirely one: there is no residual
frequency effect left to attribute to loyalty. Had the observed ratio come in
*above* 9.85x, that excess would have been the real signal worth reporting.

The over-prediction is informative in its own right. The null expects 6,975
switchers and only 2,783 exist, because a real enrolment flag is **sticky** --
once someone joins they tend to stay flagged -- while a coin re-draws on every
order. That stickiness is also what makes the before/after comparison in
`loyalty_order_rate()` meaningful, and it is the same fact
`loyalty_flag_ordering()` establishes from the other direction.

It answers **no**, and in the opposite direction to the cohort gap: customers
whose first order was on loyalty go on to average 3.6 orders against 6.3 for
everyone else, with a median of 1 against 2.

#### What is safe to quote

Three claims survive everything above, and the wording matters:

1. **A targeting fact, not a causal one.** Customers who have ever used loyalty
   average 11.69 orders (median 4) against 3.52 (median 1) for those who never
   have. Loyalty participation is *concentrated* among frequent customers. True
   and useful for sizing and targeting -- but it does not mean loyalty produced
   the frequency.
2. **No evidence the program creates frequency.** Within the same customers the
   order rate is flat before and after enrolling, 1.795 against 1.783 per 28
   days, and the first-order split runs the other way.
3. **Loyalty orders are smaller, at every frequency level.** This one holds
   under matching, which is what distinguishes it from the rest.

What the program does change is the *shape* of the distribution rather than its
average: the median order rises 9.99 -> 10.99 while p99 falls 48.81 -> 43.96,
so the mean sits still at ~13.05. Lifting the bottom and trimming the top is a
real effect that a mean alone would have reported as nothing happening.

### `customer_lifetime_value` and `customer_rfm`

20,174 rows each, one per identified customer, so peeking at rows tells you
nothing -- the distribution is what there is to read.

Note what both tables exclude by construction: `user_id` is null or blank on
12,669 orders, so anonymous trade is outside them entirely. Any customer-level
total will fall short of `daily_restaurant_sales` for that reason. It is a
caveat to state, not a bug to fix.

In [ ]:
rg.clv_segments(spark, cfg)

`customer_rfm` measures its 90-day windows from the last order in the **data**,
not from `batch_date`. The two are over two years apart here, so anchoring on
`batch_date` would file every customer as dormant. `anchor_date` is published
on the table so a reader can see which it used.

In [ ]:
anchor = rg.peek(spark, cfg, "gold", "customer_rfm", limit=1).anchor_date[0]
print(f"anchor_date: {anchor}   (batch_date is 2026-10-16)")

rg.rfm_segments(spark, cfg)

### `category_period_sales` -- three grains in one table

Week, Month and Season live together, told apart by `period_type`, because the
grains **do not nest**: weeks straddle month boundaries, so a monthly row
cannot be summed from weekly ones. They are three independent rollups of the
same orders -- which is why `orders` and `sales` agree across the three while
the row counts do not, and that agreement is the check worth reading.

`is_holiday_week` is NULL above the week grain by design, so
`null_holiday_flag` should equal `rows` for Month and Season and be 0 for Week.

In [ ]:
rg.category_grains(spark, cfg)

### `category_holiday_sales` -- against an ordinary-day baseline

`total_sales` is **not** comparable between holidays. Pooling the "Observed"
day makes `holiday_days` 5 for Christmas, New Year's Day and Independence Day
but 4 for the rest, so those three look bigger for that reason alone.
`avg_revenue_per_day` is the comparable figure, and `lift_pct` against the
ordinary-day baseline is what the table exists to show.

The helper uses `TRY_DIVIDE` rather than `/` because ANSI mode is on in Spark
4: a zero baseline would end the run instead of returning null.

In [ ]:
rg.holiday_lift(spark, cfg)

Most holidays trade **below** an ordinary day, which is the right answer for
corporate food service -- offices close, so the sites close with them. The rows
worth looking at are the minor federal holidays that are not widely taken off.

Pass a category to see one line of business on its own.

In [ ]:
rg.holiday_lift(spark, cfg, "Breakfast")

## 4. One source defect that reaches gold

Three `item_category` values arrived with a scraped admin URL concatenated into
them, and one has it spliced into the *middle* of the name
(`Drip C<url>offee`). That is worse than cosmetic: it **splits a real category
in two**, so the clean row understates its own totals and both category tables
carry the fragment as a separate row.

98 line items of 203,518, so it will not move a headline number -- but it
belongs beside any category ranking, and the fix is at the source, not in a
transform.

In [ ]:
rg.dirty_categories(spark, cfg)

## 5. The measures the dashboard needs

Total revenue, average order value, orders per day, orders per week.

Per day comes straight off the table. Per week is a rollup on `week_start`,
and the one thing to get right is that **`avg_order_total` is recomputed, not
averaged** -- a mean of means weights a 2-order Tuesday like a 200-order
Saturday. `rg.weekly()` does it as `sum(revenue) / sum(orders)`.

`sum(total_orders)` *is* safe, which is less obvious: it is a distinct count
within a day, and an order has exactly one creation timestamp, so the daily
sets are disjoint and cannot double-count.

In [ ]:
rg.weekly(spark, cfg).head(20)

## 6. How the table was written

`snapshots` is one row per write, and its `operation` column is the useful
part: `append`, `overwrite` or `delete`. This is how you confirm a
`delete_insert` actually deleted, or that a `replace` rewrote rather than
appended. The `summary` map carries added and deleted record and file counts.

In [ ]:
rg.snapshots(spark, cfg, "gold", "daily_restaurant_sales")

`partitions` answers the small-file question the Readme raises: rows and files
per partition. `months(sales_date)` should give partitions worth having rather
than hundreds holding a few KB each.

In [ ]:
rg.partitions(spark, cfg, "gold", "daily_restaurant_sales")

In [ ]:
rg.files(spark, cfg, "gold", "daily_restaurant_sales", limit=20)

## 7. Time travel

Every write leaves the previous snapshot readable, which is the honest way to
check what a transform change did to the numbers rather than reasoning about
it.

One trap worth knowing, since Spark's own error message walks you into it.
`option("snapshot-id")` was removed in Spark 4.1 and raises, telling you to use
`versionAsOf` -- which is correct, and `rg.as_of` uses it. But if you instead
reach for `snapshot_id` with an underscore, Iceberg does not recognise the
option, silently ignores it and hands back the **current** table. It returns
13,413 rows for all seven snapshots here, including the three that hold zero.
An option that raises is safe; one that is ignored produces a diff of a table
against itself.

In [ ]:
# Diff the two most recent APPENDS, not the two most recent snapshots.
#
# daily_restaurant_sales is delete_insert, so its writes come in pairs and the
# newest two snapshots are a delete and an append. The delete leaves the table
# genuinely empty -- its own manifest says total-records = 0 -- so diffing the
# last two snapshots always reads 0 -> 13,413 and tells you nothing about what
# a transform change did.
appends = rg.snapshots(spark, cfg, "gold", "daily_restaurant_sales")
appends = appends[appends.operation == "append"]

if len(appends) >= 2:
    before, after = appends.snapshot_id.iloc[-2], appends.snapshot_id.iloc[-1]
    b = rg.as_of(spark, cfg, "gold", "daily_restaurant_sales", before)
    a = rg.as_of(spark, cfg, "gold", "daily_restaurant_sales", after)
    print(f"snapshot {before}: {b.count():,} rows")
    print(f"snapshot {after}: {a.count():,} rows")
    print(f"rows only in the newer snapshot: {a.subtract(b).count():,}")
    print(f"rows only in the older snapshot: {b.subtract(a).count():,}")
else:
    print(f"need two appends to diff; found {len(appends)}")

## 8. Anything else

`rg.sql` runs arbitrary SQL with a row cap. Identifiers are three-part:
`iceberg.<layer>.<table>`, where the layer name comes from `layers:` in
config.

In [ ]:
rg.sql(spark, """
    SELECT sales_date, count(*) AS restaurants,
           sum(total_orders) AS orders,
           sum(total_revenue) AS revenue
    FROM iceberg.gold.daily_restaurant_sales
    GROUP BY sales_date
    ORDER BY sales_date DESC
""", limit=30)

In [ ]:
rg.schema(spark, cfg, "silver", "order_items")